# 🏥 MedRAX Reproduction: Safe Single-Account `gemini-3.5-flash-lite` Evaluation
### Reproducing "MedRAX: Medical Reasoning Agent for Chest X-ray" (ICML 2025)
This notebook evaluates **ChestAgentBench** using **`gemini-3.5-flash-lite`** with **Anti-Abuse Rate Pacing (Random Jitter)** and **Zero-Retry Quota Stop**:

| Feature | Design & Anti-Abuse Protection |
| :--- | :--- |
| **Model** | **`gemini-3.5-flash-lite`** (Multimodal Vision + Native Tool Calling) |
| **Zero-Retry on 429** | **`max_retries=0`** in LangChain + **0 retries on `429`** in script: The instant a quota limit is reached, traffic stops 100% (never repeats rejected requests!) |
| **Human-Like Jitter Pacer** | **Randomized Interval (`5.6s – 6.6s` = ~9.5–10 RPM)**: Safely 35% below the 15 RPM limit with organic random jitter so it never looks like a fixed-timer bot |
| **Session Question Cap** | Supports `--max-questions N` so you can stop cleanly after `N` questions *before* even hitting a single `429` |
| **Drive Sync & Resume** | **Saves every 10 questions (`os.fsync`)** + emergency flush on stop. Resume anytime without losing a single question! |


## 1. ⚙️ Hardware & GPU Environment Diagnostics
Verify the Colab runtime environment (CUDA availability, GPU model, and VRAM).
Since `gemini-3.5-flash-lite` runs on the cloud, 100% of your Colab GPU (15GB T4 or 22.5GB L4) is dedicated to MedRAX's radiology tools.


In [1]:
# Check GPU availability and specifications
import os
import sys
import torch

print(f"Python Version : {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Detected GPU   : {device_name} ({vram_gb:.2f} GB VRAM)")
    print("🚀 Local GPU is 100% dedicated to MedRAX clinical vision tools!")
else:
    print("⚠️ WARNING: No GPU detected! Please enable GPU in Runtime -> Change runtime type.")

!nvidia-smi


Python Version : 3.13.16
PyTorch Version: 2.14.1+cu130
CUDA Available : True
Detected GPU   : NVIDIA L4 (22.03 GB VRAM)
🚀 Local GPU is 100% dedicated to MedRAX clinical vision tools!
Thu Nov 19 04:30:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   64C    P8             17W /   72W |      

## 2. 💾 Google Drive Mounting & Persistent Storage
Mount Google Drive to persist downloaded datasets, model checkpoints, and evaluation results (`predictions.jsonl`).
Every 10 questions (plus any remaining questions when an account hits its daily limit) are force-synced directly to Google Drive via `os.fsync`.


In [2]:
# Mount Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/MedRAX_Thesis')

PERSISTENT_DIRS = {
    "checkpoints": DRIVE_ROOT / "checkpoints",
    "data": DRIVE_ROOT / "data" / "chestagentbench",
    "results": DRIVE_ROOT / "results",
    "logs": DRIVE_ROOT / "logs",
}

for name, path in PERSISTENT_DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"📁 Verified persistent directory: {path}")

# Set environment variables
os.environ["MEDRAX_MODEL_DIR"] = str(PERSISTENT_DIRS["checkpoints"])
os.environ["BENCHMARK_DATA_DIR"] = str(PERSISTENT_DIRS["data"])
os.environ["MEDRAX_RESULTS_DIR"] = str(PERSISTENT_DIRS["results"])
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

print("\n✅ Environment variables configured successfully.")


Mounted at /content/drive
📁 Verified persistent directory: /content/drive/MyDrive/MedRAX_Thesis/checkpoints
📁 Verified persistent directory: /content/drive/MyDrive/MedRAX_Thesis/data/chestagentbench
📁 Verified persistent directory: /content/drive/MyDrive/MedRAX_Thesis/results
📁 Verified persistent directory: /content/drive/MyDrive/MedRAX_Thesis/logs

✅ Environment variables configured successfully.


## 3. 🐙 Clone Reproduction Repository from GitHub
Clone the reproduction suite from GitHub. If already present, pull the latest code updates.


In [3]:
# Configure GitHub repository
import os
from pathlib import Path

REPO_URL = "https://github.com/Uyhuynh23/MedRAX-ChestAgentBench.git"
LOCAL_REPO_DIR = Path("/content/MedRAX-ChestAgentBench")

if not LOCAL_REPO_DIR.exists():
    print(f"Cloning {REPO_URL} into {LOCAL_REPO_DIR} ...")
    !git clone {REPO_URL} {LOCAL_REPO_DIR}
else:
    print(f"Repository exists. Pulling latest updates...")
    %cd {LOCAL_REPO_DIR}
    !git pull origin main

%cd {LOCAL_REPO_DIR}
!python scripts/setup_drive.py


Cloning https://github.com/Uyhuynh23/MedRAX-ChestAgentBench.git into /content/MedRAX-ChestAgentBench ...
Cloning into '/content/MedRAX-ChestAgentBench'...
remote: Enumerating objects: 344, done.
remote: Counting objects: 100% (344/344), done.
remote: Compressing objects: 100% (266/266), done.
remote: Total 344 (delta 170), reused 238 (delta 72), pack-reused 0 (from 0)
Receiving objects: 100% (344/344), 1.24 MiB | 18.35 MiB/s, done.
Resolving deltas: 100% (170/170), done.
/content/MedRAX-ChestAgentBench
[Drive Setup] Running in Google Colab environment.
[Drive Setup] Google Drive is already mounted.

 Initializing Persistent Google Drive Architecture
 Root: /content/drive/MyDrive/MedRAX_Thesis
  [OK] Directory ready: /content/drive/MyDrive/MedRAX_Thesis/checkpoints
  [OK] Directory ready: /content/drive/MyDrive/MedRAX_Thesis/cache/huggingface
  [OK] Directory ready: /content/drive/MyDrive/MedRAX_Thesis/cache/torch
  [OK] Directory ready: /content/drive/MyDrive/MedRAX_Thesis/data/chestag

## 4. 📦 Install Dependencies & Specialized Radiology Packages
Install PyTorch, LangChain, TorchXRayVision, Transformers, and Google Generative AI SDK.


In [4]:
%cd /content/MedRAX-ChestAgentBench

# Install project dependencies
!pip install -q -r requirements.txt
!pip install -q torchxrayvision
!pip install -q -U langchain-google-genai google-generativeai

print("\n✅ All dependencies installed successfully.")


/content/MedRAX-ChestAgentBench
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
grpc-google-iam-v1 0.14.6 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-datastore 2.27.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-connection 1.23.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 wh

## 5. 🔑 Configure Active Account API Key (1 Key & 1 IP Session at a time)

### 🛡️ Golden Rules so Google NEVER flags your account for abuse:
1. **Only 1 API Key in this session**: Do **NOT** store multiple accounts' keys in the same Colab notebook.
2. **Zero-Retry on 429**: The code is configured with `max_retries=0` and **0 retries on `429`**. The exact moment a quota limit is reached, it saves to Google Drive and stops 100% of API requests immediately (never repeating rejected requests).
3. **When switching to another Google Account**:
   * **IMPORTANT**: Do **NOT** simply paste Account #2's key into the same running Colab VM right after Account #1 hits its limit (because the running Colab VM has the same IP address!).
   * Instead, click **`Runtime -> Disconnect and delete runtime`** (Ngắt kết nối và xóa thời gian chạy) so Google Colab assigns a **brand-new VM with a new IP address** (or open the notebook in Account #2's own browser profile), and then resume from Google Drive!


In [5]:
# -------------------------------------------------------------
# 🔑 Set Active Gemini API Key (Single Account Only)
# -------------------------------------------------------------
import os
from getpass import getpass

active_key = None
try:
    from google.colab import userdata
    for candidate_secret in ["GEMINI_API_KEY", "GOOGLE_API_KEY"]:
        try:
            val = userdata.get(candidate_secret)
            if val and val.strip():
                active_key = val.split(",")[0].strip()
                print(f"🔑 Loaded key from Colab Secret [{candidate_secret}]")
                break
        except Exception:
            pass
except Exception:
    pass

if not active_key:
    active_key = getpass("Enter your single active GEMINI_API_KEY: ").strip()

if active_key:
    os.environ["GEMINI_API_KEY"] = active_key
    os.environ["GOOGLE_API_KEY"] = active_key
    masked = f"{active_key[:6]}...{active_key[-6:]}" if len(active_key) > 12 else "***"
    print(f"✅ Active API Key configured: {masked} (Zero-Retry Anti-Abuse Mode)")
else:
    print("❌ No API key provided!")


Enter your single active GEMINI_API_KEY: ··········
✅ Active API Key configured: AQ.Ab8...s300Pw (Zero-Retry Anti-Abuse Mode)


In [6]:
# -------------------------------------------------------------
# 🧪 Single-Key Vision Ground-Truth Proof & Latency Check
# -------------------------------------------------------------
# Draws secret code "7492" inside image pixels (never in text prompt)
# and verifies that gemini-3.5-flash-lite reads it accurately!
import time
import os
import google.generativeai as genai
from PIL import Image, ImageDraw

active_key = os.environ.get("GEMINI_API_KEY", "").strip()
MODEL_NAME = "gemini-3.5-flash-lite"

# Draw secret visual code "7492" onto image canvas
test_img = Image.new("RGB", (320, 120), color=(20, 20, 20))
draw = ImageDraw.Draw(test_img)
draw.rectangle([10, 10, 310, 110], outline=(220, 220, 220), width=3)
draw.text((40, 45), "SECRET CODE: 7492", fill=(255, 255, 255))

print("=======================================================")
print(f" Running Single-Key Vision & Latency Check ({MODEL_NAME})")
print("=======================================================\n")

try:
    genai.configure(api_key=active_key)
    model = genai.GenerativeModel(MODEL_NAME)
    t0 = time.perf_counter()
    resp = model.generate_content([
        "Read the exact 4-digit secret code written inside this image.",
        test_img
    ])
    elapsed = time.perf_counter() - t0
    ans = " ".join((resp.text or "").split())
    if "7492" in ans:
        print(f"  ✅ VISION PROVEN ({elapsed:.2f}s | {MODEL_NAME}) -> Read '7492' from image! Output: {ans}")
    else:
        print(f"  ⚠️ Response received ({elapsed:.2f}s): {ans}")
except Exception as e:
    print(f"  ❌ FAILED -> {e}")


/tmp/ipykernel_857/1920979795.py:8: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


 Running Single-Key Vision & Latency Check (gemini-3.5-flash-lite)

  ✅ VISION PROVEN (1.89s | gemini-3.5-flash-lite) -> Read '7492' from image! Output: 7492


## Check if out of quota or not


In [8]:
import os
from getpass import getpass
import google.generativeai as genai

# 1. Xóa chuỗi bị dán nhầm trong biến môi trường cũ
os.environ.pop("GEMINI_API_KEY", None)
os.environ.pop("GOOGLE_API_KEY", None)

# 2. Nhập lại đúng API Key (chỉ dài ~39-55 ký tự, bắt đầu bằng AIza... hoặc AQ.Ab8...)
active_key = getpass("Dán đúng GEMINI_API_KEY của bạn vào đây rồi nhấn Enter: ").strip()

os.environ["GEMINI_API_KEY"] = active_key
os.environ["GOOGLE_API_KEY"] = active_key

masked = f"{active_key[:6]}...{active_key[-6:]}" if len(active_key) > 12 else "RỖNG/SAI"
print(f"\n📌 Key mới vừa nạp: {masked} (Độ dài: {len(active_key)} ký tự)")

# 3. Kiểm tra kết nối
genai.configure(api_key=active_key)
try:
    m = genai.GenerativeModel("gemini-3.5-flash-lite")
    resp = m.generate_content("Reply with OK")
    print(f"🟢 KEY HỢP LỆ & QUOTA ĐÃ SẴN SÀNG! Phản hồi từ Gemini: {resp.text.strip()}")
except Exception as e:
    print(f"🔴 Kết quả kiểm tra:\n   -> {e}")

Dán đúng GEMINI_API_KEY của bạn vào đây rồi nhấn Enter: ··········

📌 Key mới vừa nạp: AQ.Ab8...2xBUJQ (Độ dài: 53 ký tự)
🟢 KEY HỢP LỆ & QUOTA ĐÃ SẴN SÀNG! Phản hồi từ Gemini: OK


## 6. 📥 Download ChestAgentBench Dataset & Extract Figures
Download the **ChestAgentBench** benchmark dataset (`wanglab/chest-agent-bench`) from Hugging Face into Google Drive.
If already present on Google Drive, it will verify integrity and skip re-downloading.


In [7]:
# Download / Verify ChestAgentBench in persistent Google Drive
%cd /content/MedRAX-ChestAgentBench
!python scripts/download_benchmark.py --data-dir "$BENCHMARK_DATA_DIR"

!ls -lh "$BENCHMARK_DATA_DIR"
!head -n 2 "$BENCHMARK_DATA_DIR/metadata.jsonl" | cut -c 1-120


/content/MedRAX-ChestAgentBench
 ChestAgentBench Dataset Setup
 Target Directory: /content/drive/MyDrive/MedRAX_Thesis/data/chestagentbench
[Check] Dataset files and figures directory already exist.

----------------- Verification Summary -----------------
 Total Multiple-Choice Questions: 2500 (Expected: 2,500)
 Unique Clinical Cases:           609 (Expected: ~609)
 Image References Checked:        4629/4629
 Case Folders in figures/:        957
--------------------------------------------------------
[SUCCESS] Dataset ready for MedRAX evaluation!

total 790M
drwx------ 959 root root 4.0K Nov 13 16:46 figures
-rw-------   1 root root 785M Nov 13 16:46 figures.zip
-rw-------   1 root root 4.7M Nov 13 16:46 metadata.jsonl
-rw-------   1 root root 3.4K Nov 13 16:46 README.md
{"images": ["figures/11583/figure_1.jpg"], "image_source_urls": ["https://www.eurorad.org/sites/default/files/styles/fig
{"images": ["figures/11583/figure_1.jpg"], "image_source_urls": ["https://www.eurorad.org/sites

## 7. 🔍 Check Saved Progress on Google Drive
Run the cell below anytime to check how many questions are already saved in Google Drive before starting or resuming a run.


In [9]:
# -------------------------------------------------------------
# 📊 Check Saved Progress on Google Drive (100 Cases ~ 400 Questions)
# -------------------------------------------------------------
import json
from pathlib import Path

results_dir = Path(os.environ["MEDRAX_RESULTS_DIR"])
for run_name in ["gemini_baseline_100cases", "gemini_medrax_100cases"]:
    pred_file = results_dir / run_name / "predictions.jsonl"
    if pred_file.exists():
        lines = [json.loads(line) for line in open(pred_file, encoding="utf-8") if line.strip()]
        corr = sum(1 for r in lines if r.get("is_correct"))
        acc = (corr / len(lines) * 100) if lines else 0.0
        print(f"📁 {run_name:<25}: {len(lines):>4} questions saved | Running Acc: {acc:.2f}% ({corr}/{len(lines)})")
    else:
        print(f"📁 {run_name:<25}:    0 questions saved (Not started yet)")


📁 gemini_baseline_100cases :    0 questions saved (Not started yet)
📁 gemini_medrax_100cases   :    0 questions saved (Not started yet)


## 8. 🧪 Benchmark Execution (`--max-cases 100` | Cochran 95% Confidence Subset)

### Statistical & Resource Design (`--max-cases 100` ≈ 400 Questions):
* **Cochran's Formula ($N = 2,500$, 95% Confidence, $\pm 5\%$ Margin of Error)**: Requires $n \ge 333$ questions. Evaluating **100 cases (~400 questions)** exceeds Cochran's threshold and covers all 7 clinical categories (*Detection, Classification, Localization, Comparison, Relationship, Diagnosis, Characterization*).
* **Fits in 1 Single Free Account / 1 Day**: Both Baseline (~400 calls) and MedRAX Agent (~800 calls) require only ~1,200 API calls total — fitting comfortably inside **1 single Google account** without account switching!
* **Auto-Resume & 10-Question Drive Sync**: Results are force-synced (`os.fsync`) every 10 questions. Because `--reset` is NOT passed on Option A & Option B, re-running a cell always resumes from the exact next question!


In [10]:
# -------------------------------------------------------------
# 🧪 Step 8.1a: Smoke Test - Baseline Mode on 1 Case (~15 sec)
# -------------------------------------------------------------
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py \
    --mode baseline \
    --model gemini-3.5-flash-lite \
    --data-dir "$BENCHMARK_DATA_DIR" \
    --output-dir "$MEDRAX_RESULTS_DIR/smoke_test_baseline" \
    --max-cases 1 \
    --reset


/content/MedRAX-ChestAgentBench
[Reset] --reset active: Clearing previous predictions in /content/drive/MyDrive/MedRAX_Thesis/results/smoke_test_baseline to evaluate 100% clean under gemini-3.5-flash-lite.
[Filter] Selected 5 questions across 1 cases (limit: 1).

 Starting ChestAgentBench Evaluation
 Mode: baseline | Model: gemini-3.5-flash-lite | Total Questions: 5
 Results will be saved to: /content/drive/MyDrive/MedRAX_Thesis/results/smoke_test_baseline/predictions.jsonl

/content/MedRAX-ChestAgentBench/scripts/run_evaluation.py:163: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
[1/5] QID: 11583_8353800736258 | Pred: F Gold: B ✗ | Running Acc: 0.00% (0/1)
[2/5] QID: 11583_83538008411

In [11]:
# -------------------------------------------------------------
# 🧪 Step 8.1b: Smoke Test - MedRAX Agent Mode on 1 Case (~45 sec)
# -------------------------------------------------------------
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py \
    --mode medrax \
    --model gemini-3.5-flash-lite \
    --data-dir "$BENCHMARK_DATA_DIR" \
    --output-dir "$MEDRAX_RESULTS_DIR/smoke_test_medrax" \
    --max-cases 1 \
    --reset


/content/MedRAX-ChestAgentBench
[Reset] --reset active: Clearing previous predictions in /content/drive/MyDrive/MedRAX_Thesis/results/smoke_test_medrax to evaluate 100% clean under gemini-3.5-flash-lite.
[Filter] Selected 5 questions across 1 cases (limit: 1).
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
[Init] Initializing MedRAX Agent with tools: ['ChestXRayClassifierTool', 'ChestXRaySegmentationTool', 'ChestXRayReportGeneratorTool', 'XRayVQATool'] on cuda...
If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet1

In [9]:
# -------------------------------------------------------------
# 📊 Option A: Multimodal Baseline Evaluation (100 Cases ~ 400 Questions)
# -------------------------------------------------------------
# Saves every 10 questions to Drive + immediate save on stop.
# Safe to re-run anytime (NO --reset flag, so it auto-resumes!).
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py \
    --mode baseline \
    --model gemini-3.5-flash-lite \
    --data-dir "$BENCHMARK_DATA_DIR" \
    --output-dir "$MEDRAX_RESULTS_DIR/gemini_baseline_100cases" \
    --max-cases 100


/content/MedRAX-ChestAgentBench
[Resume] Found 13 questions already evaluated. Skipping them.
[Filter] Selected 420 questions across 100 cases (limit: 100).

 Starting ChestAgentBench Evaluation
 Mode: baseline | Model: gemini-3.5-flash-lite | Total Questions: 420
 Results will be saved to: /content/drive/MyDrive/MedRAX_Thesis/results/gemini_baseline_100cases/predictions.jsonl

/content/MedRAX-ChestAgentBench/scripts/run_evaluation.py:163: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
[14/420] QID: 7603_8353801567341 | Pred: A Gold: C ✗ | Running Acc: 78.57% (11/14)
[15/420] QID: 7603_8353800862555 | Pred: B Gold: B ✓ | Running Acc: 80.00% (12/15)
[16/420] QID: 7603_8353800871132 | Pred

In [8]:
# -------------------------------------------------------------
# 🏥 Option B: MedRAX Agent Evaluation (100 Cases ~ 400 Questions)
# -------------------------------------------------------------
# Coordinates local GPU tools (CheXagent-2-3B, TorchXRayVision) + Gemini 3.5 Flash-Lite.
# Saves every 10 questions to Drive + immediate save on stop.
# Safe to re-run anytime (NO --reset flag, so it auto-resumes!).
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py \
    --mode medrax \
    --model gemini-3.5-flash-lite \
    --data-dir "$BENCHMARK_DATA_DIR" \
    --output-dir "$MEDRAX_RESULTS_DIR/gemini_medrax_100cases" \
    --max-cases 100


/content/MedRAX-ChestAgentBench
[Resume] Found 362 questions already evaluated. Skipping them.
[Filter] Selected 420 questions across 100 cases (limit: 100).
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
[Init] Initializing MedRAX Agent with tools: ['ChestXRayClassifierTool', 'ChestXRaySegmentationTool', 'ChestXRayReportGeneratorTool', 'XRayVQATool'] on cuda...
If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]
If

## 9. 📊 Comparative Analytics: Baseline vs MedRAX Agent (100 Cases)
Generate side-by-side comparison tables and accuracy breakdown charts across the 7 clinical axes:
* **Detection**, **Classification**, **Localization**, **Comparison**, **Relationship**, **Diagnosis**, **Characterization**


In [14]:
# -------------------------------------------------------------
# ⚖️ Compare Baseline vs MedRAX Agent Performance (100 Cases)
# -------------------------------------------------------------
%cd /content/MedRAX-ChestAgentBench

!python experiments/compare_runs.py \
    "$MEDRAX_RESULTS_DIR/gemini_baseline_100cases/predictions.jsonl" \
    "$MEDRAX_RESULTS_DIR/gemini_medrax_100cases/predictions.jsonl" \
    --output-dir "$MEDRAX_RESULTS_DIR/comparisons_gemini_100cases"

# Display generated comparison plot
from IPython.display import Image, display
chart_path = Path(os.environ["MEDRAX_RESULTS_DIR"]) / "comparisons_gemini_100cases" / "accuracy_breakdown.png"
if chart_path.exists():
    display(Image(str(chart_path)))


/content/MedRAX-ChestAgentBench
usage: compare_runs.py [-h] [--seed SEED] files [files ...]
compare_runs.py: error: unrecognized arguments: --output-dir /content/drive/MyDrive/MedRAX_Thesis/results/comparisons_gemini_100cases


## 10. 🔄 Synchronize Results & Updates to GitHub
Push any notebook changes or evaluation summaries back to your GitHub repository.


In [15]:
# Configure Git credentials in Colab
!git config --global user.name "Uy Huynh"
!git config --global user.email "huynhtanuy123@gmail.com"

# Check Git status
%cd /content/MedRAX-ChestAgentBench
!git status -s


/content/MedRAX-ChestAgentBench
?? cache
?? checkpoints
?? logs
?? results
